In [2]:
import chess
import random as rd
import math
from dataclasses import dataclass
class Bot:
    name: str = "bot"

    def select_move(self, board):
        raise NotImplementedError

    def __repr__(self):
         return f"<{type(self).__name__} name={self.name!r}>"

class RandomBot(Bot):

    name = "random"

    def __init__(self, seed):
        self.rng = rd.Random(seed)

    def select_move(self, board):
        moves = list(board.legal_moves)
        if not moves:
            raise ValueError("no legal move available in this position")
        return self.rng.choice(moves)



pvalues: dict[chess.PieceType, int] = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}

def ismate(board, move):
    board.push(move)
    r = board.is_checkmate()
    board.pop()
    return r

def move_gain(board, move):
    gain = 0
    if ismate(board,move):
        gain += 100
    if board.is_en_passant(move):
        gain += pvalues[chess.PAWN]

    else:
        taken = board.piece_at(move.to_square)
        if taken is not None:
            gain += pvalues[taken.piece_type]
        if move.promotion is not None:
            gain+= pvalues[move.promotion] - pvalues[chess.PAWN]
    return gain

class materialBot(Bot):

    name = "material"

    def __init__(self, seed):
        self.rng= rd.Random(seed)

    def select_move(self, board):
        moves = list(board.legal_moves) 
        if not moves:
            raise ValueError("no legal move available in this position")
        best = max(move_gain(board,m) for m in moves)
        best_moves = [m for m in moves if move_gain(board, m)==best]
        return self.rng.choice(best_moves)

In [3]:
DRAW = 0.5

In [4]:
class MatchResult:
    def __init__(self, bot_a, bot_b, games, wins_a, draws, wins_b):
        self.bot_a = bot_a
        self.bot_b = bot_b
        self.games = games
        self.wins_a = wins_a
        self.draws = draws
        self.wins_b = wins_b
    @property
    def scoreA(self):
        score = self.wins_a + self.draws*DRAW
        return score
    @property
    def scoreB(self):
        score = self.wins_b + self.draws*DRAW
        return score

    def __str__(self):
        return( f"{self.bot_a} vs {self.bot_b}: {self.scoreA} - {self.scoreB}")



In [ ]:
def play_game(white, black, max_moves=400):
    b = chess.Board()
    moves= 0
    while moves < max_moves and b.outcome(claim_draw=True) is None:
        bot = white if b.turn == chess.WHITE else black
        moves +=1
        m = bot.select_move(b)
        if m not in b.legal_moves:
            raise ValueError(f"{bot.name} returned an illegal move: {m}")
        b.push(m)
    return b

def game_score(board, botiswhite):
    outcome = board.outcome(claim_draw=True)
    if outcome is None or outcome.winner is None:
        return DRAW
    else:
        return 1.0 if outcome.winner==botiswhite else 0.0

In [68]:
rand = RandomBot(42)
rand = materialBot(484)

In [75]:

b = play_game(rand, mat)
game_score(b, True)

0.5

In [79]:
def play_match(botA, botB, nbmatchs, maxmoves=400):
    wins_A = 0
    wins_B= 0
    draws = 0
    bots = [botA,botB]
    for i in range(nbmatchs):
        AIsWhite = i%2 ==0
        white, black = (botA, botB) if AIsWhite else (botB, botA)
        b = play_game(white,black, maxmoves)
        score = game_score(b, AIsWhite)
        if score == 0.5:
            draws+= 1
        elif score ==1:
            wins_A+=1
        else:
            wins_B+=1

    return(MatchResult(botA.name, botB.name, nbmatchs, wins_A, draws, wins_B))


In [80]:
botA = RandomBot(34)
botB = materialBot(41)
str(play_match(botA, botB, 1000))

'random vs material: 61.5 - 938.5'